# Case 1: representative dataset for main text Figure 2

Generate the Section 4.1 dataset with N=2000, D=20, K*=10, component
standard deviation 1, and data seed 1. Leave features unstandardized.
Fit K-means with seed 42 for K=2,...,25 and save the ordinary Silhouette
and external scores used in Figure 2's ordinary-Silhouette panel.

The fixed r=5,30,100 comparisons and standalone ACC/ARI/NMI plots here
are additional diagnostics. `combined_boxplot_figure.ipynb` selects r
with Algorithm 1 and assembles the published Figure 2.


In [ ]:
# -- Core numerical / plotting stack --
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch

# -- Statistics: Hungarian algorithm (for clustering accuracy) and Spearman rank correlation --
from scipy.optimize import linear_sum_assignment
from scipy.stats import spearmanr

# -- Clustering (K-means), synthetic Gaussian-mixture data generation, and external validity metrics --
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

# -- k-nearest-neighbor queries used to build the rNN affinity distance matrices --
from sklearn.neighbors import NearestNeighbors

## Helper functions

In [ ]:
def simulate_gaussian_data(n_samples=2000, n_features=20, K_star=10,
                           cluster_std=1, random_state=1):
    """Generate raw Case 1 Gaussian-mixture data (main text Section 4.1).
    
    The study uses N=2000, D=20, K*=10 and component standard deviation 1.
    Centers are drawn uniformly from [-10, 10]^20 by make_blobs. Seed 1
    defines the representative dataset; seeds 0--99 define the replications.
    Return raw features. The representative plots use them directly;
    the 100-replication boxplot caller standardizes each generated dataset."""
    # Main text Section 4.1: draw centers uniformly from [-10,10]^20 and
    # X_i | y_i=c ~ N(mu_c, I_20). With N=2000 and K*=10, each component has 200 points.
    # Changing random_state redraws the centers, observations, and their row order.
    X, y = make_blobs(n_samples=n_samples, n_features=n_features,
                      centers=K_star, cluster_std=cluster_std,
                      random_state=random_state)
    return X, y


def clustering_accuracy(y_true, y_pred):
    """Clustering accuracy via the Hungarian algorithm.

    Cluster labels produced by K-means are arbitrary integers with no
    inherent correspondence to the true class labels, so "accuracy" first
    requires finding the best 1-to-1 matching between predicted clusters
    and true classes. `w[i, j]` counts how many points fall in predicted
    cluster i AND true class j; `linear_sum_assignment` (Hungarian
    algorithm) finds the permutation that maximizes the total matched
    count (it solves a minimization problem, hence `w.max() - w` as the
    cost matrix). Accuracy is that best matched count divided by the
    total number of points.
    """
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    D = max(y_pred.max(), y_true.max()) + 1
    w = np.zeros((D, D), dtype=np.int64)
    for i in range(y_pred.size):
        w[y_pred[i], y_true[i]] += 1
    # ACC = max over one-to-one label matchings of (matched observations)/N.
    # The assignment solver minimizes cost, so negation or max-count subtraction
    # converts maximum overlap to its cost convention. Unmatched groups earn no credit.
    row_ind, col_ind = linear_sum_assignment(w.max() - w)
    return w[row_ind, col_ind].sum() / y_pred.size


def rnn_affinity_distance(X, r):
    """Build the binary-neighbor distance used by rNN-Silhouette.
    
    Query r+1 neighbors and discard the first returned match. For distinct
    observations this leaves r other points. Take the symmetric union of
    directed neighbor links and return 1-A with a zero diagonal. Reuse this
    distance matrix across candidate cluster counts for the same dataset."""
    n = X.shape[0]
    # For distinct observations, dropping the first match removes self.
    # (Package-wide convention: r = number of NON-SELF neighbors, hence
    # n_neighbors=r+1 followed by dropping column 0 -- sklearn always
    # returns each point itself, at distance 0, as its own first neighbor.)
    nbrs = NearestNeighbors(n_neighbors=r + 1, algorithm='auto').fit(X)
    dists, idx = nbrs.kneighbors(X)
    # With distinct points, column 0 is the query itself, leaving r other neighbors.
    # For coincident points, the neighbor library determines the order of zero-distance ties.
    dists, idx = dists[:, 1:], idx[:, 1:]

    # Section 3.1: first form the directed binary affinity B_ij = 1{j in N_r(i)}.
    # Neighbor search uses features only; neither the reference labels nor K enter B.
    A = np.zeros((n, n))
    for i in range(n):
        A[i, idx[i]] = 1.0

    # Symmetric union: A_ij = max(B_ij, B_ji) = 1{j in N_r(i) OR i in N_r(j)}.
    # An edge in either direction is enough; this is not a mutual-neighbor graph.
    A = np.maximum(A, A.T)   # symmetrise (OR rule): i~j if EITHER direction lists the other
    np.fill_diagonal(A, 0)

    # Normalize the symmetric binary affinity before converting it to distance.
    # The affinity is already binary. This retained min-max step leaves its 0/1 values unchanged.
    a_min, a_max = A.min(), A.max()
    if a_max > a_min:
        A = (A - a_min) / (a_max - a_min)
    # The paper uses dissimilarities: d_r(i,j)=1-A_ij off the diagonal, d_r(i,i)=0.
    # Thus linked points have distance 0 and unlinked points have distance 1.
    # Pass this N-by-N matrix as metric="precomputed", not as N feature vectors.
    D = 1.0 - A
    np.fill_diagonal(D, 0)
    return D

## Experimental settings

In [ ]:
N           = 2000       # number of samples
d           = 20         # ambient dimensionality
K_STAR      = 10         # true number of Gaussian clusters (ground-truth K*)
CLUSTER_STD = 1          # per-cluster isotropic standard deviation (make_blobs)
RANDOM_STATE = 1         # seed for the representative dataset used throughout this notebook

R_VALUES = [5, 30, 100]          # neighbourhood sizes r (fixed, illustrative -- unrelated to the K-range convention below)

def max_K_eval(k_star):
    """Return K-hat = max(25, round(1.75 K*)) for the benchmark evaluation grid.
    
    Reported rank correlations use candidate counts K=2,...,K-hat. K*
    sets this experimental range; Algorithm 1 then selects from internal
    score curves without comparing them with the reference labels."""
    # K* specifies the benchmark range only. Each K is a requested partition size;
    # the experiment assesses its ranking and does not return an estimated best K.
    return int(max(25, round(1.75 * k_star)))

K_MAX  = max_K_eval(K_STAR)
K_TEST = list(range(2, K_MAX + 1))    # K = 2 ... K_MAX -- the K-means sweep evaluated below

print(f'Data : N={N}, d={d}, K_*={K_STAR}, std={CLUSTER_STD}')
print(f'r    : {R_VALUES}')
print(f'K_MAX = max(25, 1.75*K_*) = {K_MAX}')
print(f'K test: {K_TEST}')

## Generate data

In [ ]:
# Generate the single representative dataset used throughout this notebook's
# figures. random_state=1 makes this reproducible, and is the exact seed
# that combined_boxplot_figure.ipynb re-derives (via the same
# simulate_gaussian_data call) when it later needs this identical dataset
# to compute r*, since this notebook itself never computes r*.
X, y_true = simulate_gaussian_data(
    n_samples=N, n_features=d, K_star=K_STAR,
    cluster_std=CLUSTER_STD, random_state=RANDOM_STATE)
print(f'X: {X.shape},  unique labels: {np.unique(y_true)}')

## Pre-compute rNN affinity distance matrices

In [ ]:
# Build binary neighbor distances once per r and reuse them across K.
# For partition C_K, a_i is mean d_r within its cluster (excluding self),
# b_i is the smallest mean d_r to another cluster, and
# S_r(K) = mean_i[(b_i-a_i)/max(a_i,b_i)], with 0 for a_i=b_i=0 or a singleton.
# Vary r to change d_r while retaining C_K; each array scores one r across K.
dist_hard = {}
for r in R_VALUES:
    print(f'r={r}  hard ...', end=' ')
    dist_hard[r] = rnn_affinity_distance(X, r=r)
    print('done')
print('All distance matrices ready.')

## K-sweep: collect all metrics

In [ ]:
def run_k_sweep(X, y_true, K_test, dist_hard_dict, r_values, random_state=42):
    """Fit K-means at each candidate K and score the ordinary and rNN Silhouettes.
    
    The neighbor-distance dictionaries are computed once per dataset. Save
    ACC, ARI, and NMI alongside internal curves for rank comparisons. Invalid
    single-cluster partitions receive NaN internal scores."""
    res = dict(K_test=K_test, sil_original=[], accuracy=[], ari=[], nmi=[])
    for r in r_values:
        res[f'sil_hard_r{r}'] = []

    # Ordinary Silhouette uses Euclidean distances; rNN-Sil uses binary d_r.
    # Both average the same pointwise Silhouette formula over the same partition.
    for K in K_test:
        labels = KMeans(n_clusters=K, random_state=random_state,
                        n_init='auto').fit_predict(X)
        n_uniq = len(np.unique(labels))

        # External validity metrics only need the predicted labels (not a
        # distance matrix), so they are always computable regardless of
        # whether the clustering degenerated to fewer than K clusters.
        res['accuracy'].append(clustering_accuracy(y_true, labels))
        res['ari'].append(adjusted_rand_score(y_true, labels))
        res['nmi'].append(normalized_mutual_info_score(y_true, labels))

        if n_uniq > 1:
            res['sil_original'].append(
                silhouette_score(X, labels, metric='euclidean'))
            for r in r_values:
                res[f'sil_hard_r{r}'].append(
                    silhouette_score(dist_hard_dict[r], labels,
                                    metric='precomputed'))
        else:
            # Only one cluster came out -- silhouette is undefined here.
            res['sil_original'].append(np.nan)
            for r in r_values:
                res[f'sil_hard_r{r}'].append(np.nan)

    for key in res:
        if key != 'K_test':
            res[key] = np.array(res[key])
    return res


print('Running K-sweep ...')
results = run_k_sweep(X, y_true, K_TEST, dist_hard_dict=dist_hard, r_values=R_VALUES)
print('Done.')

In [ ]:
# Cache the K-sweep results to disk. combined_boxplot_figure.ipynb loads
# this exact pickle later (byte-identical reuse) to build the left panel of
# its combined figure, rather than recomputing the K-sweep.
import pickle, os

os.makedirs('data', exist_ok=True)
with open('data/gaussian_simulation_results_paperkmax_buffer10_nonself.pkl', 'wb') as f:
    pickle.dump({'results': results, 'R_VALUES': R_VALUES,
                 'K_TEST': K_TEST, 'K_STAR': K_STAR,
                 'K_MAX': K_MAX,
                 'N': N, 'd': d}, f)
print('Results saved to data/gaussian_simulation_results_paperkmax_buffer10_nonself.pkl')

## Plotting utilities

In [ ]:
import matplotlib.ticker as mticker

def get_point_colors(K_test, K_star):
    """Map each tested K to a color that encodes closeness to true K_star:
    blue shades for K < K_star (under-partition), orange shades for
    K > K_star (over-partition). Color intensity increases the closer K is
    to K_star, so points near the truth are visually more prominent."""
    K_arr     = np.array(K_test)
    # Use distance from K* only for color intensity, not for computing validity scores.
    # Darker points correspond to partition counts closer to the reference count.
    dist      = np.abs(K_arr - K_star).astype(float)
    max_d     = dist.max() if dist.max() > 0 else 1.0
    intensity = 1.0 - dist / max_d
    # Blue denotes K<K* and orange K>K*. The plotting helper handles the best-score star.
    colors = []
    for i, K in enumerate(K_arr):
        v = 0.3 + 0.7 * intensity[i]   # keep colors within a visible range (avoid near-white/near-black extremes)
        colors.append(plt.cm.Blues(v) if K < K_star else plt.cm.Oranges(v))
    return colors, intensity


def _add_k_labels(ax, xs, ys, K_vals,
                  inside_fs=7, outside_fs=7.5,
                  threshold=0.03, offset_frac=0.13,
                  overrides=None):
    """
    Place K number labels with overlap avoidance.
      - Non-overlapping: white bold text centred on the marker.
      - Overlapping: black bold text offset in the repulsion direction,
        connected back to the marker with a thin grey line.

    overrides : dict {k_value -> None | (fx, fy)}
      None     -> force the label inside the marker (white text).
      (fx, fy) -> force the label outside in the given normalised direction.
                  Canonical directions:
                    middle-bottom  (0,   -1)
                    middle-left   (-1,    0)
                    middle-right  ( 1,    0)
                    top            (0,    1)
                    bottom-right  ( 0.707,-0.707)
                    bottom-left  (-0.707,-0.707)
    """
    n = len(xs)
    if n == 0:
        return

    # Normalize point coordinates into [0, 1] axes-fraction space so the
    # collision `threshold` distance is independent of the data's actual
    # scale/units (silhouette scores and external metrics live on very
    # different numeric ranges).
    # Normalize each axis before measuring proximity, so label placement does
    # not depend on whether the index and external scores have different units.
    xlim  = ax.get_xlim()
    ylim  = ax.get_ylim()
    xspan = max(xlim[1] - xlim[0], 1e-9)
    yspan = max(ylim[1] - ylim[0], 1e-9)

    xn = (xs - xlim[0]) / xspan
    yn = (ys - ylim[0]) / yspan

    # Pass 1: flag any point whose label would collide with a nearby
    # point's label (points closer than `threshold` in normalized space).
    # Flag close marker pairs for displaced labels; isolated labels remain centered.
    needs_offset = set()
    for i in range(n):
        for j in range(i + 1, n):
            if np.hypot(xn[i] - xn[j], yn[i] - yn[j]) < threshold:
                needs_offset.add(i)
                needs_offset.add(j)

    # For each flagged point, compute a "repulsion" direction: sum the
    # unit vectors pointing away from every nearby point (within
    # threshold*5) and normalize. This pushes the offset label away from
    # the local cluster of markers rather than in an arbitrary direction.
    # For crowded points, sum directions away from nearby markers and normalize.
    # If the directions cancel, alternate up/down to separate coincident labels.
    offset_dir = {}
    for i in sorted(needs_offset):
        fx, fy = 0.0, 0.0
        for j in range(n):
            if j == i:
                continue
            dx, dy = xn[i] - xn[j], yn[i] - yn[j]
            d = np.hypot(dx, dy)
            if 0 < d < threshold * 5:
                fx += dx / d
                fy += dy / d
        norm = np.hypot(fx, fy)
        if norm > 1e-6:
            fx, fy = fx / norm, fy / norm
        else:
            # No nearby points to repel away from -- fall back to
            # alternating up/down so co-located labels still separate.
            fx, fy = (0.0, 1.0) if i % 2 == 0 else (0.0, -1.0)
        offset_dir[i] = (fx, fy)

    # Manual overrides take priority over the automatic decision: force a
    # specific K's label inside its marker (None) or outside in a
    # hand-picked direction, for the handful of cases where the automatic
    # repulsion heuristic still produced a crowded/ugly placement.
    # Apply figure-specific layout corrections last. None centers a label; a vector
    # sets its displacement direction. Neither changes the plotted score coordinates.
    if overrides:
        for i in range(n):
            k = int(K_vals[i])
            if k not in overrides:
                continue
            direction = overrides[k]
            if direction is None:
                needs_offset.discard(i)
                offset_dir.pop(i, None)
            else:
                needs_offset.add(i)
                offset_dir[i] = direction

    for i in range(n):
        x, y, k = float(xs[i]), float(ys[i]), int(K_vals[i])
        if i not in needs_offset:
            # No collision (or overridden to "inside"): draw the K number
            # directly on top of the marker in white bold text.
            ax.text(x, y, str(k),
                    ha='center', va='center',
                    fontsize=inside_fs, fontweight='bold',
                    color='white', zorder=6)
        else:
            # Collision: draw the label offset away from the marker, with
            # a thin connecting line back to it (annotate-style callout).
            fx, fy = offset_dir[i]
            tx = x + fx * offset_frac * xspan
            ty = y + fy * offset_frac * yspan
            ax.annotate(
                str(k),
                xy=(x, y), xytext=(tx, ty),
                ha='center', va='center',
                fontsize=outside_fs, fontweight='bold',
                color='black',
                arrowprops=dict(
                    arrowstyle='-', color='gray',
                    lw=0.6, shrinkA=0, shrinkB=4,
                ),
                zorder=6,
            )


def plot_panel(ax, sil, ext, K_test, K_star, r_label, ext_label,
               x_label='Silhouette Score',
               title_fs=11, label_fs=10, tick_fs=9,
               label_overrides=None):
    """Draw one scatter panel: silhouette-type score (x-axis) vs an
    external validity metric (y-axis), one point per tested K.
    Under-partitions (K < K_star) are blue circles, over-partitions
    (K > K_star) are orange squares, and the K achieving the best external
    metric value is marked with a red star. The panel title reports the
    Spearman rank correlation between the x-axis score and the external
    metric -- the central diagnostic this whole simulation is built
    around: higher rho means the internal (silhouette-type) score ranks
    candidate K's more like the external ground-truth metric does."""
    K_arr             = np.array(K_test)
    colors, intensity = get_point_colors(K_test, K_star)
    # Keep paired, non-NaN internal/external scores so each visible point
    # refers to the same partition K on both axes.
    valid_all         = ~np.isnan(sil) & ~np.isnan(ext)

    # Draw under- and over-partitioned regions separately. Lines follow increasing
    # K within each region; the markers show the individual candidate partitions.
    for mask, cmap_fn, marker, ms in [
        (K_arr <  K_star, plt.cm.Blues,   'o', 200),
        (K_arr >  K_star, plt.cm.Oranges, 's', 260),  # K=K_star excluded: no square
    ]:
        # Keep paired, non-NaN internal/external scores so each visible point
        # refers to the same partition K on both axes.
        valid = mask & valid_all
        if not np.any(valid):
            continue
        c_valid = [colors[i] for i in np.where(valid)[0]]
        ax.plot(sil[valid], ext[valid],
                lw=1.5, color=cmap_fn(0.5), alpha=0.3, zorder=1)
        ax.scatter(sil[valid], ext[valid],
                   c=c_valid, s=ms, marker=marker,
                   zorder=3, edgecolors='white', linewidths=0.4)

    # The red star marks the largest external score in this K sweep.
    if valid_all.any():
        # The star identifies the maximum external score as a retrospective reference.
        # It is not Algorithm 1's output: Algorithm 1 selects r, not this best K.
        best_i = int(np.nanargmax(ext))
        ax.scatter(sil[best_i], ext[best_i],
                   marker='*', s=350, color='red',
                   zorder=5, edgecolors='darkred', linewidths=0.5)

    _add_k_labels(ax, sil[valid_all], ext[valid_all], K_arr[valid_all],
                  overrides=label_overrides)

    # Report rank correlation across K, using the score arrays themselves.
    # Marker positions, color choices, and label offsets do not enter this calculation.
    rho      = spearmanr(sil[valid_all], ext[valid_all])[0] \
               if valid_all.sum() >= 2 else np.nan
    best_val = np.nanmax(ext)
    best_K   = K_arr[int(np.nanargmax(ext))]

    ax.set_title(
        f'{r_label}\n'
        f'Spearman $\\rho$: {rho:.3f},  '
        f'Best {ext_label}: {best_val:.3f} ($\\mathbf{{K}}$={best_K})',
        fontsize=title_fs, fontweight='bold', pad=4)
    ax.set_xlabel(x_label, fontsize=label_fs)
    ax.set_ylabel(ext_label, fontsize=label_fs)
    ax.tick_params(labelsize=tick_fs)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
    ax.grid(True, linestyle=':', alpha=0.4)


def build_legend_handles():
    """Proxy legend entries (best-K star / under-partition circle /
    over-partition square) shared across every panel in a figure."""
    # Create legend-only artists with the same marker styles as the plotted points.
    # They explain best external score, K<K*, and K>K* without adding observations.
    from matplotlib.lines import Line2D
    return [
        Line2D([0], [0], marker='*', color='none', markerfacecolor='red',
               markeredgecolor='darkred', markersize=15,
               label='Best $K$'),
        Line2D([0], [0], marker='o', color='none', markerfacecolor='steelblue',
               markeredgecolor='white', markersize=12,
               label='$K <$ True $K^*$  (underestimate)'),
        Line2D([0], [0], marker='s', color='none', markerfacecolor='darkorange',
               markeredgecolor='white', markersize=13,
               label='$K >$ True $K^*$  (overestimate)'),
    ]


def _merge(base, extra):
    """Merge two flat override dicts; extra takes priority."""
    # Copy the base mapping first so panel-specific overrides do not mutate shared defaults.
    d = dict(base or {})
    d.update(extra or {})
    return d


def _merge_subplot(base_sub, extra_sub):
    """
    Merge two subplot-override dicts {(row,col): {k: dir}}.
    Per-subplot K entries are merged; extra takes priority.
    """
    # Copy each nested panel mapping before merging its K-specific label overrides.
    result = {key: dict(val) for key, val in (base_sub or {}).items()}
    for key, kdict in (extra_sub or {}).items():
        if key in result:
            result[key].update(kdict)
        else:
            result[key] = dict(kdict)
    return result


def make_figure(results, r_values, K_test, K_star,
                ext_key, ext_label, figsize=(16, 9),
                figure_overrides=None,
                figure_original_overrides=None):
    """Draw fixed-r diagnostic panels above the ordinary-Silhouette comparison.
    
    The first row has one rNN panel per neighborhood size. The second row
    contains ordinary Silhouette and a common legend. Label-position overrides
    affect the layout only."""
    # -- Font sizes -----------------------------------------------------------
    fs_suptitle  = 14
    fs_row_title = 14
    fs_subtitle  = 11
    fs_axlabel   = 10
    fs_tick      = 10
    fs_legend    = 15
    # -------------------------------------------------------------------------

    # Global: applied to every subplot.
    # K=K_star: no circle/square, so force label inside the red star (white text).
    global_overrides = {
        K_test[-1]: None,   # last tested K always inside its square
        K_star:     None,   # true K* label centred on the red star
    }

    # Built-in per-subplot overrides (shared between both figures)
    base_subplot_overrides = {(0, 0): {13: (-1.0, 0.0)}, (0, 1): {12: None, 13: None, 15: None, 18: (0.0, -1.0)}, (0, 2): {12: None, 13: None, 18: (0.0, -1.0)}}

    subplot_overrides = _merge_subplot(base_subplot_overrides, figure_overrides)

    # Ordinary-Silhouette panel
    original_overrides = _merge(global_overrides, {14: None})
    original_overrides = _merge(original_overrides, figure_original_overrides)

    # All panels use this same external-score vector; only the internal-score
    # vector changes between fixed-r and ordinary-Silhouette comparisons.
    ext   = results[ext_key]
    ncols = len(r_values)

    fig = plt.figure(figsize=figsize)
    gs  = gridspec.GridSpec(2, ncols, figure=fig, hspace=0.75, wspace=0.18)

    # --- Row 0: Neighborhood Hard ---
    row0_axes = []
    for col, r in enumerate(r_values):
        ax  = fig.add_subplot(gs[0, col])
        ovr = _merge(global_overrides, subplot_overrides.get((0, col)))
        plot_panel(ax, results[f'sil_hard_r{r}'], ext,
                   K_test, K_star, f'$r = {r}$', ext_label,
                   x_label='Silhouette Score (Neighborhood Hard)',
                   title_fs=fs_subtitle, label_fs=fs_axlabel, tick_fs=fs_tick,
                   label_overrides=ovr)
        row0_axes.append(ax)

    # --- Row 1: Ordinary silhouette and legend ---
    ax_orig = fig.add_subplot(gs[1, 0])
    plot_panel(ax_orig, results['sil_original'], ext,
               K_test, K_star, 'Silhouette Score (Original)', ext_label,
               x_label='Silhouette Score (Original)',
               title_fs=fs_subtitle, label_fs=fs_axlabel, tick_fs=fs_tick,
               label_overrides=original_overrides)

    ax_leg = fig.add_subplot(gs[1, 1:])
    ax_leg.axis('off')
    ax_leg.legend(
        handles=build_legend_handles(),
        loc='center', ncol=1, fontsize=fs_legend, framealpha=0.9,
        handletextpad=0.6, columnspacing=1.5, handlelength=1.0,
    )

    fig.suptitle(
        f'Simulated Gaussian Dataset: Clustering Metrics vs {ext_label}\n'
        f'(True $\\mathbf{{K}} = {K_star}$)',
        fontsize=fs_suptitle, fontweight='bold',
    )
    fig.subplots_adjust(top=0.87)

    for axes_row, row_title in [(row0_axes, 'Silhouette Score (Neighborhood Hard)')]:
        pos0 = axes_row[0].get_position()
        posN = axes_row[-1].get_position()
        x_center = (pos0.x0 + posN.x1) / 2
        y_top    = pos0.y1
        fig.text(x_center, y_top + 0.03, row_title,
                 ha='center', va='bottom', fontsize=fs_row_title,
                 fontweight='bold', transform=fig.transFigure)

    return fig

## Additional diagnostic: ACC

In [ ]:
# Set optional K-label positions to keep scatter labels readable.
# The overrides affect only the plot layout, not scores or correlations.
fig_acc = make_figure(
    results, R_VALUES, K_TEST, K_STAR,
    ext_key='accuracy', ext_label='ACC',
    figure_overrides={(0, 0): {9: (0.0, 1.0), 15: (0.3, -0.954)}, (0, 1): {7: (0.0, 1.0), 11: (0.0, 1.0), 14: (0.0, 1.0), 19: (0.707, -0.707)}, (0, 2): {18: (0.707, -0.707), 14: (0.0, -1.0), 15: (0.0, 1.0)}},
)
import os
os.makedirs('figures/paperkmax_buffer10', exist_ok=True)
fig_acc.savefig('figures/paperkmax_buffer10/SimGa_vs_ACC.pdf', dpi=150, bbox_inches='tight')
plt.show()

## Additional diagnostic: ARI

In [ ]:
# Same idea as the ACC figure above, but with hand-tuned label overrides for
# the ARI comparison figure (a different external metric produces a
# different scatter layout, hence a different set of collisions to fix).
fig_ari = make_figure(
    results, R_VALUES, K_TEST, K_STAR,
    ext_key='ari', ext_label='ARI',
    figure_overrides={(0, 0): {14: (0.3, -0.954), 15: (0.0, -1.0)}, (0, 1): {7: (0.707, -0.707), 18: (-1.0, 0.0), 19: (0.707, -0.707)}, (0, 2): {14: None, 15: (-0.707, 0.707)}},
    figure_original_overrides={13: (0.0, -1.0)},
)
import os
os.makedirs('figures/paperkmax_buffer10', exist_ok=True)
fig_ari.savefig('figures/paperkmax_buffer10/SimGa_vs_ARI.pdf', dpi=150, bbox_inches='tight')
plt.show()

## Additional diagnostic: NMI

In [ ]:
# NMI comparison figure -- no figure-specific label overrides were needed
# here; the built-in global/subplot overrides inside make_figure() plus the
# automatic repulsion heuristic in _add_k_labels were sufficient for this
# metric's point layout.
fig_nmi = make_figure(
    results, R_VALUES, K_TEST, K_STAR,
    ext_key='nmi', ext_label='NMI',
)
import os
os.makedirs('figures/paperkmax_buffer10', exist_ok=True)
fig_nmi.savefig('figures/paperkmax_buffer10/SimGa_vs_NMI.pdf', dpi=150, bbox_inches='tight')
plt.show()

## Summary table (Spearman correlations)

In [ ]:
# Build a tidy summary table of Spearman rank correlations between each
# score variant (ordinary Silhouette, and rNN-based Hard Silhouette at
# each fixed r in R_VALUES) and each external validity metric (ACC, ARI,
# NMI). These are the same rho values already printed in each figure's
# panel titles above, collected here into one DataFrame for easy
# side-by-side comparison / export.
import pandas as pd

rows = []
for ext_key, ext_label in [('accuracy', 'ACC'), ('ari', 'ARI'), ('nmi', 'NMI')]:
    ext = results[ext_key]
    sil = results['sil_original']
    m   = ~np.isnan(sil) & ~np.isnan(ext)
    rho = spearmanr(sil[m], ext[m])[0] if m.sum() >= 2 else np.nan
    rows.append({'Method': 'Silhouette Score', 'r': '—',
                 'External': ext_label, 'Spearman rho': round(rho, 4)})
    for r in R_VALUES:
        for variant, lbl in [('hard', 'Neighborhood Hard')]:
            sil = results[f'sil_{variant}_r{r}']
            m   = ~np.isnan(sil) & ~np.isnan(ext)
            rho = spearmanr(sil[m], ext[m])[0] if m.sum() >= 2 else np.nan
            rows.append({'Method': f'Silhouette Score ({lbl})',
                         'r': r, 'External': ext_label,
                         'Spearman rho': round(rho, 4)})

df = pd.DataFrame(rows)
print(df.to_string(index=False))